In [1]:
import random
import ipywidgets as widgets                                  # the widget library (buttons, labels, boxes)
from IPython.display import display, clear_output             # display() shows widgets; clear_output() erases printed text
from IPython.display import Image as IPImage                  # `as` gives it a unique name, so PIL's Image can't replace it
import requests
from PIL import Image
import io
                                                              # audio libraries
import json                                                   
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from IPython.display import Audio
import base64


###################################################################################################

### AUDIO SETUP ###

# Load credentials from the service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

# Fetch an access token (this is what gcloud auth print-access-token returns)
creds.refresh(Request())
token = creds.token

with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

def theme_song_audio():

    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"Theme song for a trivia game about animal sidekicks in tv shows"
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    music_databytes = data["outputs"][2]['data']
    mp3_bytes = base64.b64decode(music_databytes)
    
    return Audio(mp3_bytes)

In [3]:
### LOADING UI ###

loading = widgets.HTML("""
<style>
  .theme-loader {
    width: 22px; height: 22px;
    border: 3px solid #ddd;
    border-top-color: #555;
    border-radius: 50%;
    animation: theme-spin 0.9s linear infinite;
  }
  @keyframes theme-spin { to { transform: rotate(360deg); } }
</style>
<div style="display:flex; align-items:center; gap:10px; padding:8px 0;">
  <div class="theme-loader"></div>
  <span>Composing your theme song…</span>
</div>
""")

player_area = widgets.Output()       # where the Audio player will go
song_box = widgets.VBox([])          # the slot we swap things in and out of
display(song_box)


def show_theme_song():
    song_box.children = [loading]            # 1. show spinner right away
    try:
        audio = theme_song_audio()           # 2. slow part (API call + decode)
        player_area.clear_output()
        with player_area:
            display(audio)                   # 3. render player into the Output widget
        song_box.children = [player_area]    # 4. swap spinner out, player in
    except Exception as e:
        song_box.children = [widgets.HTML(f"<b>Couldn't load the song:</b> {e}")]


show_theme_song()

VBox()